In [1]:
%pwd

'e:\\ml\\DataScience_Project1\\research'

In [2]:
import os
os.chdir("../")
%pwd

'e:\\ml\\DataScience_Project1'

In [3]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path

    


In [4]:
from src.datascience.constants import *
from src.datascience.utils.common import read_yaml, create_directories

In [5]:
class ConfigurationManager:
    def __init__(self,config_filepath=CONFIG_FILE_PATH,
                 params_filepath=PARAMS_FILE_PATH,
                 schema_filepath=SCHEMA_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config=self.config.data_ingestion
        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(root_dir=config.root_dir,
                                                    source_URL=config.source_URL,
                                                    local_data_file=config.local_data_file,
                                                    unzip_dir=config.unzip_dir)
        return data_ingestion_config

        

In [6]:
#Component data ingestion

from src.datascience.utils import logger
import zipfile
import urllib.request

In [7]:
from src.datascience.utils.logger import logging
import os
import urllib.request
import zipfile

class DataIngestion:
    def __init__(self, config):
        self.config = config

    def download_data(self):
        if not os.path.exists(self.config.local_data_file):
            logging.info(f"Downloading data from {self.config.source_URL} to {self.config.local_data_file}")
            urllib.request.urlretrieve(self.config.source_URL, self.config.local_data_file)
            logging.info("Data downloaded successfully.")

    def unzip_data(self):
        if not os.path.exists(self.config.unzip_dir):
            os.makedirs(self.config.unzip_dir)

        logging.info(f"Unzipping data from {self.config.local_data_file} to {self.config.unzip_dir}")

        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(self.config.unzip_dir)

        logging.info("Data unzipped successfully.")

In [8]:
try:
    config_manager = ConfigurationManager()
    data_ingestion_config = config_manager.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_data()
    data_ingestion.unzip_data()

except Exception as e:
    raise e

[2026-03-18 18:14:02,597] 29 root - INFO - YAML file loaded successfully from: config\config.yaml
[2026-03-18 18:14:02,603] 29 root - INFO - YAML file loaded successfully from: params.yaml
[2026-03-18 18:14:02,604] 29 root - INFO - YAML file loaded successfully from: schema.yaml
[2026-03-18 18:14:02,605] 52 root - INFO - Created directory at: artifacts
[2026-03-18 18:14:02,607] 52 root - INFO - Created directory at: artifacts/data_ingestion
[2026-03-18 18:14:02,608] 12 root - INFO - Downloading data from https://archive.ics.uci.edu/static/public/186/wine+quality.zip to artifacts/data_ingestion/data.zip
[2026-03-18 18:14:07,128] 14 root - INFO - Data downloaded successfully.
[2026-03-18 18:14:07,130] 20 root - INFO - Unzipping data from artifacts/data_ingestion/data.zip to artifacts/data_ingestion
[2026-03-18 18:14:07,342] 25 root - INFO - Data unzipped successfully.
